# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process a FAIR^2 Croissant dataset using the `mlcroissant` library, referencing all data entities by their `@id`. Examples follow the recommended Croissant and `mlcroissant` patterns for reproducible, FAIR data science.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

We first load the dataset metadata and examine its high-level description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object (do not subscript or iterate)
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Citation: {dataset.metadata.cite_as}")

## 2. Data Overview

Let's inspect the available record sets in the dataset. Each record set, field, and column is referenced by its Croissant `@id`. We'll display available record sets and the associated field `@id`s.

In [ ]:
# List all record sets with their @id and fields
record_sets = [r for r in dataset.record_sets]
print("Record sets available in this dataset:")
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id} ; name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - field @id: {field.id} ; name: {field.name}")
    print()

## 3. Data Extraction

We will extract data from each record set into a pandas DataFrame. Each record set will be referenced by its exact `@id`. For demonstration, we'll focus on the main tabular record set (use the correct `@id` printed in the previous cell, typically the first or only one for primary data).

In [ ]:
# Gather all record set @ids for reference
record_set_ids = [r.id for r in dataset.record_sets]

# We'll demonstrate using the first tabular record set (most datasets have just one)
main_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
for rec_id in record_set_ids:
    # Extract all records for each record set
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records for record set: {rec_id}")

# For illustration, show column names and first few records of main table
if main_record_set_id:
    print(f"\nColumns in the main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We now perform some basic exploratory operations. We'll reference a numeric field and categorical field using their exact `@id`.

Common EDA steps include filtering, normalization, removing outliers, and grouping/categorization.

In [ ]:
# Pick a numeric field and a group field by their @id.
# Adjust the @ids below based on field listing in Section 2 output, e.g.:
# numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/<some_field_id>'
# group_field_id = 'https://api.app.sen.science/frontiers/7862866/<some_other_field_id>'

# For demonstration purposes, let's scan for suitable numeric and group fields
df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None

# Try to find likely candidates by dtype and unique values.
for col in df.columns:
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    # As a placeholder pick a non-numeric field with a small number of unique values
    if group_field_id is None and not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < (len(df)//4):
        group_field_id = col

print(f"Chosen numeric_field_id: {numeric_field_id}")
print(f"Chosen group_field_id: {group_field_id}")

# Set a threshold (adjust as appropriate for the actual data; here, use the median)
threshold = df[numeric_field_id].median() if numeric_field_id else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization (z-score) for the filtered numeric field
if len(filtered_df) > 0:
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())
else:
    print("No records passed the filter.")

# Grouped analysis by group_field_id, if present
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Let's plot the distribution of the numeric field and perhaps a boxplot by the chosen group field (all referencing `@id` columns).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

# Boxplot by group field (if available and not too many groups)
if group_field_id and group_field_id in df.columns and df[group_field_id].nunique() < 15:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we explored a clinical oncology dataset defined by a Croissant schema and loaded using the `mlcroissant` library.
We demonstrated metadata exploration, listing of schemas and fields by `@id`, DataFrame extraction, and simple EDA with normalization and group summaries. Visualizations further revealed the distributions in the dataset.

All dataset components were referenced by their immutable identifiers (`@id`), ensuring fully reproducible and transparent science over this FAIR^2 dataset.